# Official StableMate verification

Pinned author implementation; synthetic regression data only. No cohort modeling.

In [ ]:
from pathlib import Path
import json, os, subprocess, shutil, sys
from datetime import datetime, timezone

try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None:
    drive.mount('/content/drive')

locations = []
if os.environ.get('CMPS355_ROOT'):
    locations.append(Path(os.environ['CMPS355_ROOT']).expanduser())
locations.extend([Path('/content/drive/MyDrive/CMPS355_Research'), Path.cwd(), *Path.cwd().parents])
ROOT = next((p.resolve() for p in locations if (p / '.cmps355_root').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('CMPS355_Research project root not found')
SCRIPT = ROOT / 'r' / 'verify_stablemate.R'
if not SCRIPT.is_file():
    raise FileNotFoundError(SCRIPT)
RSCRIPT = shutil.which('Rscript')
if RSCRIPT is None:
    raise RuntimeError('Rscript is required')
R_LIB = Path('/content/cmps355_r_lib') if drive is not None else ROOT / 'runs' / 'r_library'
R_LIB.mkdir(parents=True, exist_ok=True)
SHA = 'cf0fe9f344c1756b6f89fd33f9fdae1a7f35b24a'
run_dir = ROOT / 'runs' / 'official_r'
run_dir.mkdir(parents=True, exist_ok=True)
print('Project:', ROOT)
print('R:', subprocess.check_output([RSCRIPT,'--version'],stderr=subprocess.STDOUT,text=True).strip())
print('Pinned StableMate revision:', SHA)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project: /content/drive/MyDrive/CMPS355_Research
R: Rscript (R) version 4.6.1 (2026-06-24)
Pinned StableMate revision: cf0fe9f344c1756b6f89fd33f9fdae1a7f35b24a


In [ ]:
def execute_stage(mode, timeout_seconds):
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    log_path = run_dir / f'stablemate_{mode}_{stamp}.log'
    environment = os.environ.copy()
    environment['CMPS355_R_LIB'] = str(R_LIB)
    with log_path.open('w', encoding='utf-8') as handle:
        try:
            completed = subprocess.run(
                [RSCRIPT, str(SCRIPT), mode, SHA],
                stdout=handle, stderr=subprocess.STDOUT, env=environment,
                timeout=timeout_seconds, check=False
            )
            exit_code = completed.returncode
        except subprocess.TimeoutExpired:
            exit_code = -1
            handle.write('\nEXECUTION_TIMEOUT\n')
    lines = log_path.read_text(encoding='utf-8', errors='replace').splitlines()
    print(mode, 'exit code:', exit_code)
    print('Log:', log_path.relative_to(ROOT))
    print('\n'.join(lines[-24:]))
    return {'mode': mode, 'exit_code': exit_code, 'log': str(log_path.relative_to(ROOT)),
            'pinned_revision': SHA, 'verified_sha': any(line == f'SOURCE_SHA={SHA}' for line in lines),
            'smoke_pass': any(line == 'SMOKE_FIT=PASS' for line in lines)}

installation = execute_stage('install', timeout_seconds=5400)
if installation['exit_code'] != 0 or not installation['verified_sha']:
    raise RuntimeError('Pinned StableMate installation did not pass source verification; inspect the saved log')


install exit code: 0
Log: runs/official_r/stablemate_install_20261106T221432526714Z.log
BLAS:   /usr/lib/x86_64-linux-gnu/openblas-pthread/libblas.so.3 
LAPACK: /usr/lib/x86_64-linux-gnu/openblas-pthread/libopenblasp-r0.3.26.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

time zone: Etc/UTC
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

loaded via a namespace (and not attached):
 [1] desc_1.4.3       R6_2.6.1         Matrix_1.7-6     lattice_0.23-1  
 [5] magrittr_2.0.5   remotes_2.5.0    ps_1.9.3         cli_3.6.6       
 [9] processx_3.9.0   callr_3.8.0      grid_4.

In [4]:
verification = execute_stage('smoke', timeout_seconds=1800)
status = 'PASS_AUTHOR_PACKAGE_SYNTHETIC_SMOKE' if (verification['exit_code'] == 0 and verification['verified_sha'] and verification['smoke_pass']) else 'FAIL'
report = {
    'method': 'StableMate',
    'source': 'https://github.com/meiosis97/StableMate',
    'commit': SHA,
    'data': 'synthetic_regression',
    'installation': installation,
    'smoke': verification,
    'status': status,
    'created_utc': datetime.now(timezone.utc).isoformat(),
}
report_path = run_dir / f'stablemate_report_{datetime.now(timezone.utc):%Y%m%dT%H%M%S%fZ}.json'
report_path.write_text(json.dumps(report, indent=2), encoding='utf-8')
print('Report:', report_path.relative_to(ROOT))
print('Status:', status)
if status == 'FAIL':
    raise RuntimeError('Official StableMate smoke test failed; inspect the saved log')


smoke exit code: 0
Log: runs/official_r/stablemate_smoke_20261106T222258340800Z.log

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/openblas-pthread/libblas.so.3 
LAPACK: /usr/lib/x86_64-linux-gnu/openblas-pthread/libopenblasp-r0.3.26.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

time zone: Etc/UTC
tzcode source: system (glibc)

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

loaded via a namespace (and not attached):
 [1] compiler_4.6.1    magrittr_2.0.5    Matrix_1.7-6      StableMate_0.1.0 
 [5] parallel_4.6.1    codetools_0.2-20  doParallel_1.0.17 grid_4.6.1       
 [9] iterators_1